In [1]:
import sqlite3
import pandas as pd

In [2]:
%load_ext sql
%sql sqlite:///rappi_sql_practice.db

In [3]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table';

 * sqlite:///rappi_sql_practice.db
Done.


name
cities
customers
restaurants
couriers
products
orders
order_items
payments
ratings


In [4]:
%%sql
PRAGMA table_info(restaurants);

 * sqlite:///rappi_sql_practice.db
Done.


cid,name,type,notnull,dflt_value,pk
0,restaurant_id,INTEGER,0,None,1
1,restaurant_name,TEXT,1,None,0
2,city_id,INTEGER,1,None,0
3,category,TEXT,1,None,0
4,is_active,INTEGER,1,1,0
5,opened_date,DATE,1,None,0


## Pregunta 1

Devuélveme:

status

número de órdenes

ordenado de mayor a menor

In [6]:
%%sql
SELECT * FROM orders LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


order_id,customer_id,restaurant_id,courier_id,order_ts,status,delivery_fee,promo_code
1,3,4,None,2026-01-06 20:57:00,created,6.0,None


In [9]:
%%sql
SELECT status, count(*) as numero_ordenes
FROM orders
GROUP BY status
ORDER BY numero_ordenes DESC;

 * sqlite:///rappi_sql_practice.db
Done.


status,numero_ordenes
delivered,15
picked_up,14
assigned,10
paid,9
created,6
cancelled,6


## Pregunta 2
Devuélveme:

El revenue total global (solo órdenes pagadas).

Debe devolver una sola fila con:

total_revenue

Escríbela.

In [10]:
%%sql
SELECT p.amount FROM payments p LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


amount
111.0
46.9
204.2
14.0
83.5


In [12]:
%%sql
SELECT SUM(p.amount) as total_revenue
FROM payments p
WHERE p.paid_ts IS NOT NULL;


 * sqlite:///rappi_sql_practice.db
Done.


total_revenue
3892.8


## Pregunta 3

¿Cuál es el ticket promedio global?
(Solo órdenes pagadas)

Debe devolver una sola fila con:

avg_ticket

In [ ]:
%%sql
SELECT AVG(p.amount) as avg_ticket
FROM payments p
WHERE p.paid_ts IS NOT NULL
GROUP BY 

 * sqlite:///rappi_sql_practice.db
Done.


avg_ticket
81.10000000000001


## Pregunta 4

Devuélveme los Top 5 restaurantes por revenue total (solo órdenes pagadas).

Debe mostrar:

restaurant_name

total_revenue

Ordenado de mayor a menor
Solo los primeros 5.

In [20]:
%%sql

SELECT * FROM payments LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


payment_id,order_id,method,amount,paid_ts
1,1,cash,111.0,None


In [24]:
%%sql
SELECT r.restaurant_name, SUM(p.amount) as total_revenue
FROM payments p
INNER JOIN orders o ON o.order_id = p.order_id
INNER JOIN restaurants r ON r.restaurant_id = o.restaurant_id
WHERE p.paid_ts IS NOT NULL
GROUP BY r.restaurant_name
ORDER BY total_revenue DESC
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name,total_revenue
Green Bowl,946.5
Sushi Nori,927.5
Pizza Porto,882.3999999999999
Arepa & Co,428.9
Burger Lab,371.0


## Pregunta 5
¿Cuál es la ciudad con más órdenes?
(No revenue, solo cantidad de órdenes.)

Debe mostrar:

city_name

total_orders

Ordenado de mayor a menor
Solo la primera.


In [31]:
%%sql
SELECT * FROM restaurants LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_id,restaurant_name,city_id,category,is_active,opened_date
1,Arepa & Co,1,colombian,1,2022-08-15


In [32]:
%%sql 
SELECT c.name, COUNT(*) as numero_ordenes
FROM cities c 
INNER JOIN restaurants r ON r.city_id = c.city_id
INNER JOIN orders o ON o.restaurant_id = r.restaurant_id
GROUP BY c.name
ORDER BY numero_ordenes DESC
LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


name,numero_ordenes
Medellín,26


## Pregunta 6

Devuélveme el número total de órdenes por restaurante.

Debe mostrar:

restaurant_name

total_orders

Ordenado de mayor a menor.

In [33]:
%%sql
SELECT * FROM restaurants LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_id,restaurant_name,city_id,category,is_active,opened_date
1,Arepa & Co,1,colombian,1,2022-08-15


In [38]:
%%sql
SELECT r.restaurant_name, COUNT(*) as numero_ordenes
FROM orders o
INNER JOIN restaurants r ON r.restaurant_id = o.restaurant_id
GROUP BY r.restaurant_name
ORDER BY numero_ordenes DESC;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name,numero_ordenes
Green Bowl,15
Pizza Porto,11
Burger Lab,10
Taco Loco,9
Sushi Nori,8
Arepa & Co,7


## Pregunta 7
Devuélveme el total de órdenes entregadas por restaurante.

Debe mostrar:

restaurant_name

delivered_orders

Ordenados de mayor a menor

In [39]:
%%sql
SELECT r.restaurant_name, count(*) as delivered_orders
FROM orders o
INNER JOIN restaurants r ON r.restaurant_id = o.restaurant_id
WHERE o.status = 'delivered'
GROUP BY r.restaurant_name
ORDER BY delivered_orders DESC;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name,delivered_orders
Pizza Porto,5
Green Bowl,3
Taco Loco,2
Sushi Nori,2
Arepa & Co,2
Burger Lab,1


## Pregunta 8

Devuélveme el total de órdenes por ciudad (ciudad del cliente).

Debe mostrar:

city_name

total_orders

Ordenado de mayor a menor.

In [43]:
%%sql
SELECT * FROM customers LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


customer_id,full_name,email,city_id,signup_date,is_premium
1,Sofía Zarruk,sofia@example.com,1,2024-08-15,1


In [44]:
%%sql
SELECT ci.name, count(*) as total_orders
FROM cities ci
INNER JOIN customers cu ON cu.city_id = ci.city_id
INNER JOIN orders o ON cu.customer_id = o.customer_id
GROUP BY ci.name
ORDER BY total_orders DESC;

 * sqlite:///rappi_sql_practice.db
Done.


name,total_orders
Medellín,25
Bogotá,14
Cali,12
Barranquilla,9


## Pregunta 9

Devuélveme para cada ciudad:

total_orders

total_revenue (solo pagadas)

Ordenado de mayor a menor

In [47]:
%%sql
SELECT * FROM payments LIMIT 1,1;

 * sqlite:///rappi_sql_practice.db
Done.


payment_id,order_id,method,amount,paid_ts
2,2,cash,46.9,None


In [50]:
%%sql
SELECT c.name, count(*) as total_orders, SUM(p.amount) as total_revenue
FROM cities c
INNER JOIN customers cu ON cu.city_id = c.city_id
INNER JOIN orders o ON cu.customer_id = o.customer_id
INNER JOIN payments p ON p.order_id = o.order_id
WHERE p.paid_ts IS NOT NULL
GROUP BY c.name
ORDER BY total_revenue DESC;


 * sqlite:///rappi_sql_practice.db
Done.


name,total_orders,total_revenue
Medellín,21,1606.4999999999998
Bogotá,11,1428.9
Cali,9,476.5
Barranquilla,7,380.9


In [ ]:
%%sql
SELECT c.name,
    COUNT(o.order_id) AS total_orders,
    SUM(CASE WHEN p.paid_ts IS NOT NULL THEN p.amount ELSE 0 END) AS total_revenue
FROM cities c
JOIN customers cu ON cu.city_id = c.city_id
JOIN orders o ON cu.customer_id = o.customer_id
LEFT JOIN payments p ON p.order_id = o.order_id
GROUP BY c.name
ORDER BY total_revenue DESC;

 * sqlite:///rappi_sql_practice.db
Done.


name,total_orders,total_revenue
Medellín,25,1606.5
Bogotá,14,1428.9
Cali,12,476.5
Barranquilla,9,380.9


## Pregunta 10
Devuélveme los restaurantes que tienen más de 5 órdenes.

Debe mostrar:

restaurant_name

total_orders

Usa HAVING.

In [55]:
%%sql
SELECT r.restaurant_name, count(o.order_id) as total_orders
FROM orders o
INNER JOIN restaurants r ON r.restaurant_id = o.restaurant_id
GROUP BY r.restaurant_name
HAVING total_orders > 5
ORDER BY total_orders DESC;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name,total_orders
Green Bowl,15
Pizza Porto,11
Burger Lab,10
Taco Loco,9
Sushi Nori,8
Arepa & Co,7


In [57]:
%%sql
SELECT r.restaurant_name, count(o.order_id) as total_orders
FROM orders o
INNER JOIN restaurants r ON r.restaurant_id = o.restaurant_id
GROUP BY r.restaurant_name
HAVING count(o.order_id) > 5
ORDER BY total_orders DESC;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name,total_orders
Green Bowl,15
Pizza Porto,11
Burger Lab,10
Taco Loco,9
Sushi Nori,8
Arepa & Co,7


En postgres no puedo usar la variable en el HAVING

## Pregunta 11

Devuélveme los restaurantes que NO tienen órdenes.

Debe mostrar:

restaurant_name

Aquí sí necesitas LEFT JOIN + NULL.

In [64]:
%%sql
SELECT r.restaurant_name
FROM restaurants r
LEFT JOIN orders o ON o.restaurant_id = r.restaurant_id
WHERE o.order_id IS NULL;


 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name


## Pregunta 12

Devuélveme los clientes que han hecho órdenes pero nunca han pagado ninguna.

(Pista: necesitarás LEFT JOIN y revisar NULL en payments)

In [73]:
%%sql
SELECT * FROM payments LIMIT 1;

 * sqlite:///rappi_sql_practice.db
Done.


payment_id,order_id,method,amount,paid_ts
1,1,cash,111.0,None


In [72]:
%%sql
SELECT cu.full_name
FROM customers cu
INNER JOIN orders o ON o.customer_id = cu.customer_id
WHERE o.status <> 'paid'
GROUP by cu.full_name;

 * sqlite:///rappi_sql_practice.db
Done.


full_name
Andrés Silva
Camila Rojas
Diego Castro
Juan Pérez
María Gómez
Sofía Zarruk
Valentina Torres
